In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lpad, concat_ws, avg, expr, regexp_replace, round as spark_round

spark = SparkSession.builder.getOrCreate()

# Leer el fichero
df = spark.read.option("header", True) \
               .option("sep", ";") \
               .option("inferSchema", False) \
               .csv("work/calidad_aire_datos_meteo_mes.csv")

# Filtrar magnitud 83 (temperatura media)
df_temp = df.filter(col("MAGNITUD") == "83")

# Crear fecha YYYY-MM-DD
df_temp = df_temp.withColumn(
    "FECHA",
    concat_ws(
        "-",
        col("ANO"),
        lpad(col("MES"), 2, "0"),
        lpad(col("DIA"), 2, "0")
    )
)

# Convertir H01..H24 y V01..V24 en filas
df_horas = df_temp.select(
    "FECHA",
    "MUNICIPIO",
    "ESTACION",
    expr("""
        stack(24,
            '01', H01, V01,
            '02', H02, V02,
            '03', H03, V03,
            '04', H04, V04,
            '05', H05, V05,
            '06', H06, V06,
            '07', H07, V07,
            '08', H08, V08,
            '09', H09, V09,
            '10', H10, V10,
            '11', H11, V11,
            '12', H12, V12,
            '13', H13, V13,
            '14', H14, V14,
            '15', H15, V15,
            '16', H16, V16,
            '17', H17, V17,
            '18', H18, V18,
            '19', H19, V19,
            '20', H20, V20,
            '21', H21, V21,
            '22', H22, V22,
            '23', H23, V23,
            '24', H24, V24
        ) as (HORA, TEMP, VALIDACION)
    """)
)

# Quedarse solo con horas válidas y convertir temperatura a double
df_validas = df_horas.filter(col("VALIDACION") == "V") \
    .withColumn("TEMP", regexp_replace(col("TEMP"), ",", ".")) \
    .withColumn("TEMP", expr("try_cast(TEMP as double)")) \
    .filter(col("TEMP").isNotNull())

# Estación de referencia: municipio 6, estación 4
df_ref = df_validas.filter(
    (col("MUNICIPIO") == "6") & (col("ESTACION") == "4")
).groupBy("FECHA").agg(
    avg("TEMP").alias("TEMP_REF")
)

# Estación de comparación: municipio 5, estación 2
df_cmp = df_validas.filter(
    (col("MUNICIPIO") == "5") & (col("ESTACION") == "2")
).groupBy("FECHA").agg(
    avg("TEMP").alias("TEMP_CMP")
)

# Unir por fecha y calcular porcentaje respecto a la referencia
r = df_ref.alias("r")
c = df_cmp.alias("c")

resultado = r.join(
    c,
    col("r.FECHA") == col("c.FECHA"),
    "inner"
).select(
    col("r.FECHA").alias("FECHA"),
    spark_round((col("c.TEMP_CMP") / col("r.TEMP_REF")) * 100, 2).alias("PORCENTAJE")
).orderBy("FECHA")

print("Comparacion diaria de temperatura media (referencia: municipio 6, estacion 4):")
resultado.show(truncate=False)